#Introduction
This notebook documents every step—from creating a text corpus, uploading to HDFS, writing Java MapReduce code, building the JAR, running the job.<br>**Note:** Commands are highlighted, they aren't supposed to be executed in Notebook, use terminal (Linux) for the same.

#PreRequisites
  -Hadoop (3.4.1) installed and configured in single‑node mode.
    [Refer Documentation](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-common/SingleCluster.html)
  
  -Java 8 and JAVA_HOME set.
  
  -$HADOOP_HOME/bin on your PATH.
  
  -Linux preferred.

#Steps to Run the WordCount on HDFS

##Start Hadoop Daemons
`start-all.sh`
<br>The above shell script will run two another shell scripts at once,


1.   start‑dfs.sh: launches NameNode, DataNode, SecondaryNameNode
2.   start‑yarn.sh: launches ResourceManager, NodeManager, JobHistoryServer


##Verify Daemons have Started<br>
```jps```<br>It should output as,<br><pre><font color="#26A269"><b>karunya-chavan@mavrix</b></font>:<font color="#12488B"><b>~</b></font>$ jps
26993 ResourceManager
27123 NodeManager
26403 NameNode
26549 DataNode
26791 SecondaryNameNode
27502 Jps

##Prepare a Local Dataset

*   Create a plain‐text file named input.txt in your working directory.



##Upload the File(input.txt created above) to HDFS

###Create HDFS directories (with parents):
`hdfs dfs -mkdir -p /user/yourusername/wordcount/input
`
<br>-p : It indicates that parent directories will be created automatically it they doesn't exist.<br><b>Note: </b>Replace the yourusername appropriately.

###Copy Local File to HDFS
`hdfs dfs -put input.txt /user/yourusername/wordcount/input
`
<br>-put: copy from local → HDFS

###Verify
`hdfs dfs -ls /user/yourusername/wordcount/input`

##Create Java Source Files

###Directory Structure<br>
WordCount/<br>
├── src/<br>
│   └──wordcount/<br>
│       ├── WordCountMapper.java<br>
│       ├── WordCountReducer.java<br>
│       └── WordCountDriver.java<br>


###WordCountMapper.java<br>
```
package wordcount;

import java.io.IOException;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;

public class WordCountMapper extends Mapper<LongWritable, Text, Text, IntWritable> {

    private final static IntWritable one = new IntWritable(1);
    private Text word = new Text();

    @Override
    protected void map(LongWritable key, Text value, Context context)
          throws IOException, InterruptedException {
        String[] tokens = value.toString().split("\\s+");
        for (String token : tokens) {
            String cleaned = token.replaceAll("[^a-zA-Z]", "").toLowerCase();
            if (!cleaned.isEmpty()) {
                word.set(cleaned);
                context.write(word, one);
            }
        }
    }
}
```

###WordCountReducer.java<br>
```
package wordcount;

import java.io.IOException;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;

public class WordCountReducer extends Reducer<Text, IntWritable, Text, IntWritable> {

    @Override
    protected void reduce(Text key, Iterable<IntWritable> values, Context context)
          throws IOException, InterruptedException {
        int sum = 0;
        for (IntWritable val : values) {
            sum += val.get();
        }
        context.write(key, new IntWritable(sum));
    }
}

```

###WordCountDriver.java<br>
```
package wordcount;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class WordCountDriver {
    public static void main(String[] args) throws Exception {
        if (args.length != 2) {
            System.err.println("Usage: WordCount <input path> <output path>");
            System.exit(-1);
        }

        Configuration conf = new Configuration();
        Job job = Job.getInstance(conf, "Word Count");
        job.setJarByClass(WordCountDriver.class);

        job.setMapperClass(WordCountMapper.class);
        job.setReducerClass(WordCountReducer.class);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(IntWritable.class);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}
```

##Build & Package the JAR

###Compile Source<br>
`cd WordCount`<br>
`mkdir build`<br>
``javac -classpath `hadoop classpath` -d build src/wordcount/*.java``<br>

###Create JAR file<br>
`jar -cvf WordCount.jar -C build/ .`<br>
-c : create JAR<br>
-v : verbose<br>
-f : specify file name<br>

##Run the MapReduce Job<br>
```
hadoop jar WordCount.jar wordcount.WordCountDriver \
  /user/yourusername/wordcount/input \
  /user/yourusername/wordcount/output

```
<br>Note: The output directory must not exist or the job will fail.

##View Results<br>
```
hdfs dfs -ls /user/yourusername/wordcount/output```<br>```
hdfs dfs -cat /user/yourusername/wordcount/output/part-r-00000
```<br>
part‑r‑00000: output from reducer #0

##Stop Hadoop and Yarn Daemons<br>```stop-all.sh```

#Clean Up HDFS for a Fresh Start (Optional)

##Delete the Output Directory<br>
```hdfs dfs -rm -r /user/yourusername/wordcount/output```<br>
```hdfs dfs -rm -r /user/yourusername/wordcount/input
```<br>
```hdfs dfs -ls /user/yourusername/wordcount
```<br>
— should list nothing if both were removed.

#Appendix

<table>
  <thead>
    <tr>
      <th>Command</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>hdfs dfs -mkdir -p /path/to/dir</code></td>
      <td>Create directory (and parents if needed)</td>
    </tr>
    <tr>
      <td><code>hdfs dfs -put localfile /path/in/hdfs</code></td>
      <td>Copy local file into HDFS</td>
    </tr>
    <tr>
      <td><code>hdfs dfs -ls /path/in/hdfs</code></td>
      <td>List contents of an HDFS directory</td>
    </tr>
    <tr>
      <td><code>hdfs dfs -cat /path/in/hdfs/filename</code></td>
      <td>Print HDFS file contents to terminal</td>
    </tr>
    <tr>
      <td><code>hdfs dfs -rm -r /path/in/hdfs</code></td>
      <td>Recursively delete HDFS directory or files</td>
    </tr>
    <tr>
      <td><code>hadoop jar your.jar main.Class /in /out</code></td>
      <td>Submit a MapReduce job JAR (in: input path, out: output path)</td>
    </tr>
  </tbody>
</table>
